# GAN Super-Resolution Model Evaluation

Load a trained GAN checkpoint from `crop_sr_experiments.py`, generate super-resolved images, and compute common quality metrics (PSNR, SSIM, LPIPS, FID).

In [ ]:
import os
import glob
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import dataset_utils
from training_utils import (
    load_class_names,
    build_plant_dict,
    load_plant_balanced_image_file_sample,
)
from generators import ResidualGenerator
from agricultural_sr import AgriculturalAttentionGenerator, AgriculturalDiscriminator
from discriminators import SpectralNormDiscriminator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Discover & Load Checkpoint

Scans the `checkpoints/` directory for available models and loads one.

In [ ]:
CHECKPOINT_DIR = "checkpoints"

# Map experiment prefixes to their generator class and default block count
GENERATOR_MAP = {
    "agricultural": (AgriculturalAttentionGenerator, 8),
    "ag_attention_multiblock": (AgriculturalAttentionGenerator, 8),
    "residual_ag_losses": (ResidualGenerator, 4),
    "baseline_residual": (ResidualGenerator, 4),
    "loss_pixel_only": (ResidualGenerator, 4),
    "loss_perceptual_only": (ResidualGenerator, 4),
    "loss_combined": (ResidualGenerator, 4),
}

# Find all checkpoint_latest.pt files
available = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "*", "checkpoint_latest.pt")))

if available:
    print("Available checkpoints:")
    for i, p in enumerate(available):
        print(f"  [{i}] {Path(p).parent.name}")
else:
    print("No checkpoints found in checkpoints/.")
    print("Run an experiment first, e.g.:")
    print("  python crop_sr_experiments.py --experiment baseline --epochs 5")

In [ ]:
# --- Select which checkpoint to load ---
# Option 1: Set MODEL_DIR directly to a directory name (e.g. "agricultural_4blocks_total50epochs")
# Option 2: Set to None and use CHECKPOINT_IDX to pick from the list above
MODEL_DIR = None
CHECKPOINT_IDX = 0

if MODEL_DIR is not None:
    CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, MODEL_DIR, "checkpoint_latest.pt")
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
elif available:
    CHECKPOINT_PATH = available[CHECKPOINT_IDX]
else:
    raise FileNotFoundError(
        "No checkpoint found. Train a model first with crop_sr_experiments.py"
    )

model_dir_name = Path(CHECKPOINT_PATH).parent.name
print(f"Loading: {model_dir_name}")

# Parse the number of residual blocks from the directory name (e.g. '..._4blocks_...')
num_blocks = None
for part in model_dir_name.split("_"):
    if part.endswith("blocks"):
        num_blocks = int(part.replace("blocks", ""))
        break

# Determine the generator class from the directory name prefix
GeneratorClass = None
for prefix, (cls, default_blocks) in GENERATOR_MAP.items():
    if model_dir_name.startswith(prefix):
        GeneratorClass = cls
        if num_blocks is None:
            num_blocks = default_blocks
        break

if GeneratorClass is None:
    raise ValueError(
        f"Could not determine generator class for '{model_dir_name}'.\n"
        f"Known prefixes: {list(GENERATOR_MAP.keys())}"
    )

print(f"Generator: {GeneratorClass.__name__}, blocks: {num_blocks}")

# Build generator and load weights
generator = GeneratorClass(num_residual_blocks=num_blocks).to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
generator.load_state_dict(checkpoint["generator_state_dict"])
generator.eval()

epoch = checkpoint.get("epoch", "?")
metrics = checkpoint.get("metrics", {})
print(f"Checkpoint epoch: {epoch}")
if metrics:
    print("Training metrics at save time:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

## 2. Load Evaluation Data

In [ ]:
DATA_DIR = "data/plantvillage/color_images/train"
SAMPLES_PER_CLASS = 50  # small subset for evaluation

class_names = load_class_names(DATA_DIR)
plant_dict = build_plant_dict(class_names)
image_files, _, _ = load_plant_balanced_image_file_sample(
    DATA_DIR, plant_dict, SAMPLES_PER_CLASS
)

dataset = dataset_utils.SuperResolutionDataset(image_files=image_files)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, num_workers=2)
print(f"Evaluation set: {len(dataset)} images")

## 3. Generate Super-Resolved Images

Side-by-side comparison: low-res input, generated SR output, ground-truth high-res.

In [ ]:
def denormalize(tensor):
    """Convert from [-1, 1] back to [0, 1] for display."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)


def tensor_to_numpy(tensor):
    """CHW tensor in [-1,1] -> HWC numpy in [0,1]."""
    return denormalize(tensor).permute(1, 2, 0).cpu().numpy()


# Grab a batch
low_res, high_res = next(iter(dataloader))
low_res, high_res = low_res.to(device), high_res.to(device)

with torch.no_grad():
    sr_output = generator(low_res)

NUM_SHOW = min(6, low_res.size(0))
fig, axes = plt.subplots(3, NUM_SHOW, figsize=(3 * NUM_SHOW, 9))

row_labels = ["Low-Res (64x64)", "Super-Resolved (256x256)", "Ground Truth (256x256)"]
for i in range(NUM_SHOW):
    axes[0, i].imshow(tensor_to_numpy(low_res[i]))
    axes[1, i].imshow(tensor_to_numpy(sr_output[i]))
    axes[2, i].imshow(tensor_to_numpy(high_res[i]))
    for row in range(3):
        axes[row, i].axis("off")

for row, label in enumerate(row_labels):
    axes[row, 0].set_ylabel(label, fontsize=12)

plt.suptitle(f"Model: {model_dir_name}", fontsize=14)
plt.tight_layout()
plt.savefig("eval_sr_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Pixel-Level Metrics: PSNR & SSIM

Computed over the full evaluation set. We also compare against a bicubic upscale baseline.

In [ ]:
psnr_scores_sr = []
ssim_scores_sr = []
psnr_scores_bicubic = []
ssim_scores_bicubic = []

generator.eval()
with torch.no_grad():
    for low_res, high_res in dataloader:
        low_res, high_res = low_res.to(device), high_res.to(device)
        sr_output = generator(low_res)

        # Bicubic baseline: upsample low-res directly
        bicubic = F.interpolate(low_res, size=(256, 256), mode="bicubic", align_corners=False)

        for i in range(high_res.size(0)):
            gt = tensor_to_numpy(high_res[i])
            sr = tensor_to_numpy(sr_output[i])
            bc = tensor_to_numpy(bicubic[i])

            psnr_scores_sr.append(psnr(gt, sr, data_range=1.0))
            ssim_scores_sr.append(ssim(gt, sr, data_range=1.0, channel_axis=2))

            psnr_scores_bicubic.append(psnr(gt, bc, data_range=1.0))
            ssim_scores_bicubic.append(ssim(gt, bc, data_range=1.0, channel_axis=2))

print(f"{'Metric':<10} {'GAN SR':>12} {'Bicubic':>12}")
print("-" * 36)
print(f"{'PSNR (dB)':<10} {np.mean(psnr_scores_sr):>12.2f} {np.mean(psnr_scores_bicubic):>12.2f}")
print(f"{'SSIM':<10} {np.mean(ssim_scores_sr):>12.4f} {np.mean(ssim_scores_bicubic):>12.4f}")
print(f"\nEvaluated on {len(psnr_scores_sr)} images")

## 5. LPIPS (Learned Perceptual Similarity)

Lower is better. Measures perceptual distance using deep features. Requires `pip install lpips`.

In [ ]:
try:
    import lpips

    lpips_fn = lpips.LPIPS(net="alex").to(device)
    lpips_scores_sr = []
    lpips_scores_bicubic = []

    with torch.no_grad():
        for low_res, high_res in dataloader:
            low_res, high_res = low_res.to(device), high_res.to(device)
            sr_output = generator(low_res)
            bicubic = F.interpolate(
                low_res, size=(256, 256), mode="bicubic", align_corners=False
            )

            # LPIPS expects inputs in [-1, 1]
            lpips_sr = lpips_fn(sr_output, high_res).squeeze()
            lpips_bc = lpips_fn(bicubic, high_res).squeeze()

            if lpips_sr.dim() == 0:
                lpips_scores_sr.append(lpips_sr.item())
                lpips_scores_bicubic.append(lpips_bc.item())
            else:
                lpips_scores_sr.extend(lpips_sr.cpu().tolist())
                lpips_scores_bicubic.extend(lpips_bc.cpu().tolist())

    print(f"LPIPS (lower is better):")
    print(f"  GAN SR:  {np.mean(lpips_scores_sr):.4f}")
    print(f"  Bicubic: {np.mean(lpips_scores_bicubic):.4f}")

except ImportError:
    print("lpips not installed. Run: pip install lpips")
    print("Skipping LPIPS evaluation.")

## 6. FID (Frechet Inception Distance)

Compares the distribution of generated images to real images using Inception v3 features. Lower is better. Requires `pip install pytorch-fid`.

In [ ]:
from torchvision.models import inception_v3
from scipy.linalg import sqrtm


def get_inception_features(images, inception_model, device, batch_size=32):
    """Extract Inception v3 pool3 features from a list of image tensors."""
    features = []
    for i in range(0, len(images), batch_size):
        batch = torch.stack(images[i : i + batch_size]).to(device)
        # Inception v3 expects 299x299 input in [0, 1]-ish range
        batch = denormalize(batch)
        batch = F.interpolate(batch, size=(299, 299), mode="bilinear", align_corners=False)
        with torch.no_grad():
            feat = inception_model(batch)
        features.append(feat.cpu())
    return torch.cat(features, dim=0).numpy()


def calculate_fid(real_features, fake_features):
    """Compute FID between two sets of features."""
    mu_real, sigma_real = real_features.mean(axis=0), np.cov(real_features, rowvar=False)
    mu_fake, sigma_fake = fake_features.mean(axis=0), np.cov(fake_features, rowvar=False)

    diff = mu_real - mu_fake
    covmean, _ = sqrtm(sigma_real @ sigma_fake, disp=False)
    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff @ diff + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return float(fid)


# Load Inception v3 for feature extraction
inception = inception_v3(pretrained=True, transform_input=True)
inception.fc = torch.nn.Identity()  # remove final FC to get 2048-d features
inception = inception.to(device).eval()

# Collect generated and real images
real_images = []
sr_images = []
MAX_FID_IMAGES = 500  # FID is more reliable with more samples

with torch.no_grad():
    for low_res, high_res in dataloader:
        low_res = low_res.to(device)
        sr_output = generator(low_res)
        for i in range(high_res.size(0)):
            real_images.append(high_res[i])
            sr_images.append(sr_output[i].cpu())
        if len(real_images) >= MAX_FID_IMAGES:
            break

print(f"Computing FID over {len(real_images)} images...")
real_feat = get_inception_features(real_images, inception, device)
sr_feat = get_inception_features(sr_images, inception, device)

fid_score = calculate_fid(real_feat, sr_feat)
print(f"FID: {fid_score:.2f} (lower is better)")

## 7. Summary & Metric Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(psnr_scores_sr, bins=30, alpha=0.7, label="GAN SR")
axes[0].hist(psnr_scores_bicubic, bins=30, alpha=0.7, label="Bicubic")
axes[0].set_xlabel("PSNR (dB)")
axes[0].set_ylabel("Count")
axes[0].set_title("PSNR Distribution")
axes[0].legend()
axes[0].axvline(np.mean(psnr_scores_sr), color="C0", linestyle="--", alpha=0.5)
axes[0].axvline(np.mean(psnr_scores_bicubic), color="C1", linestyle="--", alpha=0.5)

axes[1].hist(ssim_scores_sr, bins=30, alpha=0.7, label="GAN SR")
axes[1].hist(ssim_scores_bicubic, bins=30, alpha=0.7, label="Bicubic")
axes[1].set_xlabel("SSIM")
axes[1].set_ylabel("Count")
axes[1].set_title("SSIM Distribution")
axes[1].legend()
axes[1].axvline(np.mean(ssim_scores_sr), color="C0", linestyle="--", alpha=0.5)
axes[1].axvline(np.mean(ssim_scores_bicubic), color="C1", linestyle="--", alpha=0.5)

plt.suptitle(f"Model: {model_dir_name}", fontsize=13)
plt.tight_layout()
plt.savefig("eval_metric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Summary ===")
print(f"Model: {model_dir_name}")
print(f"Checkpoint epoch: {epoch}")
print(f"PSNR:  {np.mean(psnr_scores_sr):.2f} dB  (bicubic: {np.mean(psnr_scores_bicubic):.2f})")
print(f"SSIM:  {np.mean(ssim_scores_sr):.4f}     (bicubic: {np.mean(ssim_scores_bicubic):.4f})")
try:
    print(f"LPIPS: {np.mean(lpips_scores_sr):.4f}     (bicubic: {np.mean(lpips_scores_bicubic):.4f})")
except NameError:
    pass
try:
    print(f"FID:   {fid_score:.2f}")
except NameError:
    pass